## module 1: improved classifier
same resnet50 setup as the 03 baseline, with fine-grained upgrades: stronger augmentation, label smoothing, discriminative learning rates (lower for backbone, higher for head), cosine schedule, more epochs.

resumable: saves full state (incl. optimizer + scheduler) every `CKPT_EVERY` steps and continues from the last checkpoint — survives a long run or a colab disconnect.

In [ ]:
import os, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# subset toggle: set N_SPECIES to an int (e.g. 100) for a quick run, None for all
N_SPECIES = None

splits = pd.read_csv("../data/splits.csv")
if N_SPECIES:
    keep = sorted(splits["species"].unique())[:N_SPECIES]
    splits = splits[splits["species"].isin(keep)].reset_index(drop=True)

labels = sorted(splits["species"].unique())
label2idx = {s:i for i,s in enumerate(labels)}

print(len(labels), "species")
print(splits["split"].value_counts().to_string())

In [ ]:
# stronger train aug than baseline: color jitter, randaugment, random erasing
MEAN = [0.485,0.456,0.406]
STD = [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.RandAugment(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
    transforms.RandomErasing(p=0.25),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

In [ ]:
class PlantSet(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        return self.tf(img), label2idx[row["species"]]

train_ds = PlantSet(splits[splits["split"]=="train"], train_tf)
val_ds = PlantSet(splits[splits["split"]=="val"], eval_tf)
test_ds = PlantSet(splits[splits["split"]=="test"], eval_tf)

# batch 16 fits a 4gb gpu (resnet50 @224); bump if you have more vram
BATCH = 16
val_dl = DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0)
test_dl = DataLoader(test_ds,batch_size=BATCH,shuffle=False,num_workers=0)
len(train_ds), len(val_ds), len(test_ds)

In [ ]:
EPOCHS = 15

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(labels))
model = model.to(device)

# discriminative lr: pretrained backbone learns slowly, fresh head learns faster
head = list(model.fc.parameters())
backbone = [p for n,p in model.named_parameters() if not n.startswith("fc.")]
optimizer = torch.optim.AdamW([
    {"params": backbone, "lr": 1e-4},
    {"params": head, "lr": 1e-3},
], weight_decay=1e-2)

# label smoothing helps calibration on noisy fine-grained labels
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def run_epoch(dl, desc):
    # eval only; training is inline in the loop so it can checkpoint mid-epoch
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x,y in tqdm(dl, desc=desc, leave=False):
            x,y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out,y).item()*len(y)
            correct += (out.argmax(1)==y).sum().item()
            n += len(y)
    return total_loss/n, correct/n

In [ ]:
# --- checkpointing ---
# COLAB: local disk is wiped on disconnect, so put CKPT_DIR on mounted Drive:
#   from google.colab import drive; drive.mount('/content/drive')
#   CKPT_DIR = '/content/drive/MyDrive/botanical-vision/checkpoints'
CKPT_DIR = "../checkpoints"
CKPT_EVERY = 1000    # save every N optimizer steps (lower = safer, more IO)
RUN_NAME = "resnet50_improved"
FRESH = False        # True = ignore any existing checkpoint and start over
SEED = 42

os.makedirs(CKPT_DIR, exist_ok=True)
last_path = f"{CKPT_DIR}/{RUN_NAME}_last.pt"
best_path = f"{CKPT_DIR}/{RUN_NAME}_best.pt"

# signature: a run only resumes a checkpoint with the SAME setup (else it's a different experiment)
sig = hashlib.md5(json.dumps({"model":"resnet50","n_species":N_SPECIES,"epochs":EPOCHS,"n_labels":len(labels)}, sort_keys=True).encode()).hexdigest()[:10]

def save_ckpt(path, epoch, step):
    # (epoch, step) = the NEXT batch to run. atomic write survives a mid-save disconnect.
    payload = {"sig":sig, "epoch":epoch, "step":step, "state_dict":model.state_dict(),
               "opt":optimizer.state_dict(), "sched":scheduler.state_dict(),
               "best_val":best_val, "hist":hist, "labels":labels}
    torch.save(payload, path+".tmp"); os.replace(path+".tmp", path)

# resume if a matching checkpoint exists, else start fresh
start_epoch, start_step, best_val = 0, 0, 0.0
hist = {"train_loss":[],"val_loss":[],"train_acc":[],"val_acc":[]}
if os.path.exists(last_path) and not FRESH:
    ck = torch.load(last_path, weights_only=False, map_location=device)
    if ck["sig"] == sig:
        model.load_state_dict(ck["state_dict"]); optimizer.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        start_epoch, start_step, best_val, hist = ck["epoch"], ck["step"], ck["best_val"], ck["hist"]
        print(f"resuming: epoch {start_epoch+1}, step {start_step}, best_val {best_val:.3f}")
    else:
        print("existing checkpoint has a different config -> starting fresh")
else:
    print("fresh run")

In [ ]:
for ep in range(start_epoch, EPOCHS):
    # deterministic per-epoch order so a mid-epoch resume continues exactly where it stopped
    g = torch.Generator().manual_seed(SEED + ep)
    perm = torch.randperm(len(train_ds), generator=g).tolist()
    skip = start_step if ep == start_epoch else 0
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=perm[skip*BATCH:], num_workers=0)
    nbatch = len(train_dl)

    model.train()
    tot, cor, seen = 0.0, 0, 0
    bar = tqdm(train_dl, desc=f"epoch {ep+1}/{EPOCHS} train", leave=False)
    for i,(x,y) in enumerate(bar):
        step = skip + i
        x,y = x.to(device), y.to(device)
        out = model(x); loss = criterion(out,y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot += loss.item()*len(y); cor += (out.argmax(1)==y).sum().item(); seen += len(y)
        bar.set_postfix(loss=f"{tot/seen:.3f}", acc=f"{cor/seen:.3f}")
        if (step+1) % CKPT_EVERY == 0 and i < nbatch-1:
            save_ckpt(last_path, ep, step+1)
    tl, ta = tot/max(seen,1), cor/max(seen,1)

    vl, va = run_epoch(val_dl, f"epoch {ep+1}/{EPOCHS} val")
    scheduler.step()   # cosine schedule advances once per epoch
    hist["train_loss"].append(tl); hist["val_loss"].append(vl)
    hist["train_acc"].append(ta); hist["val_acc"].append(va)
    if va > best_val:
        best_val = va
        save_ckpt(best_path, ep+1, 0)
    save_ckpt(last_path, ep+1, 0)   # epoch boundary
    start_step = 0
    print(f"epoch {ep+1}: train_loss {tl:.3f} acc {ta:.3f} | val_loss {vl:.3f} acc {va:.3f}")

print(f"best val acc: {best_val:.3f}")

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(12,4))
fig.set_facecolor("linen")

axs[0].plot(hist["train_loss"],color="darkorange",label="train")
axs[0].plot(hist["val_loss"],color="sienna",label="val")
axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Loss")
axs[0].set_title("Loss",weight="bold"); axs[0].legend()

axs[1].plot(hist["train_acc"],color="darkorange",label="train")
axs[1].plot(hist["val_acc"],color="sienna",label="val")
axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy",weight="bold"); axs[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# reload best-val checkpoint, then top-1 / top-5 on test
ck = torch.load(best_path, weights_only=False, map_location=device)
model.load_state_dict(ck["state_dict"]); model.eval()

top1, top5, n = 0, 0, 0
with torch.no_grad():
    for x,y in tqdm(test_dl, desc="test"):
        x,y = x.to(device), y.to(device)
        out = model(x)
        top5_pred = out.topk(5,dim=1).indices
        top1 += (out.argmax(1)==y).sum().item()
        top5 += (top5_pred==y.unsqueeze(1)).any(1).sum().item()
        n += len(y)

print(f"test top-1: {top1/n:.3f}")
print(f"test top-5: {top5/n:.3f}")